# MassGAT: Results Visualization
This notebook provides an interactive tool to explore MassGAT inference results.

**Prerequisites:** The MassGAT pipeline (IonShed → Graph Construction → Inference) must have already been executed.

## Setup: Configure Paths
Define the project workspace and point to the results from a previous pipeline run.

In [1]:
import os
from pathlib import Path

# Define the project workspace and directories
os.environ["PROJ_NAME"] = "test"  # Change this to your project name
DOC_DIR = Path(os.getcwd())
BASE_DIR = DOC_DIR.parent
DATA_DIR = BASE_DIR / "data" / os.environ["PROJ_NAME"]
GRAPHS_DIR = DATA_DIR / "graphs"
RESULTS_DIR = DATA_DIR / "results"

# Define ionization mode (must match the pipeline run)
IONIZATION_MODE = "positive"  # "positive" or "negative"

print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Graphs directory: {GRAPHS_DIR}")
print(f"Ionization mode: {IONIZATION_MODE}")

Data directory: \\fouet\spi\scidospi\08_Team\PinartPaulHenri\massgat\data\test
Results directory: \\fouet\spi\scidospi\08_Team\PinartPaulHenri\massgat\data\test\results
Graphs directory: \\fouet\spi\scidospi\08_Team\PinartPaulHenri\massgat\data\test\graphs
Ionization mode: positive


---
## Interactive Results Visualization
Use the interface below to search for a specific chemical target in your processed graphs. Enter your expected **m/z** and **Retention Time (min)** to locate the component, extract the pseudo-spectrum, and visualize MassGAT's predictions.

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd, torch, networkx as nx, glob
from pathlib import Path

from src.utils.visualizations import create_nx_graph, visualize_component

# Setup directories for visualizations
VIS_DIR = DATA_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

display(HTML("<style>.search-card{background:#fff;border:1px solid #e5e7eb;border-radius:12px;padding:16px;box-shadow:0 1px 3px rgba(0,0,0,.06);} .search-card .widget-dropdown>select,.search-card .widget-text>input,.search-card .widget-numeric-text>input{border-radius:8px!important;} .search-card .widget-button{border-radius:8px!important;}</style>"))

ui_style = {'description_width': 'initial'}
tsv_files = [Path(p).name for p in glob.glob(str(RESULTS_DIR / "*_metadata.tsv"))]

w_file = widgets.Dropdown(options=tsv_files, description='Results TSV:', style=ui_style)
w_graph = widgets.Dropdown(options=[], description='Graph (.pt):', style=ui_style)
w_mz = widgets.FloatText(value=152.0556, description='Target m/z:', step=0.0001, style=ui_style)
w_rt = widgets.FloatText(value=4.47, description='Target RT (min):', step=0.01, style=ui_style)
w_ppm = widgets.FloatText(value=5.0, description='Tol (ppm):', step=1.0, style=ui_style)
w_title = widgets.Text(value='Guanine', description='Save Title:', style=ui_style)

w_btn = widgets.Button(description='Search', button_style='primary', icon='search')
w_save = widgets.Button(description='Save HTML', button_style='success', icon='save', disabled=True)
w_match = widgets.Dropdown(options=[], description='Select Match:', style=ui_style, disabled=True)

out_plot = widgets.Output()

# Global state to cache data and avoid reloading files when switching matches
state = {'graph': None, 'G': None, 'node_probs': None}
last_args = {}

# Update graph options based on selected TSV file
def update_graph_options(*a):
    if not w_file.value: return w_graph.options.clear()
    try: w_graph.options = pd.read_csv(RESULTS_DIR / w_file.value, sep='\t', usecols=['file_name'])['file_name'].dropna().unique().tolist()
    except: w_graph.options = []

w_file.observe(update_graph_options, names='value')
update_graph_options()

# Render logic: triggers when a specific match is selected
def render_selected_node(change):
    if change['type'] != 'change' or change['name'] != 'value' or change['new'] is None: return
    
    node = change['new']
    w_save.disabled = True
    with out_plot:
        clear_output(wait=True)
        component = nx.node_connected_component(state['G'], node)
        
        print(f"✅ Rendering Node {node} (Component size: {len(component)} peaks)")
        last_args.update(dict(G=state['G'], nodes=component, highlight_nodes=[node], node_probs=state['node_probs'], 
                              graph_name=w_graph.value, ionization_mode=IONIZATION_MODE, remove_low_prob=True,
                              use_prob_styling=True, show_profiles=True, 
                              graph=state['graph'], style="presentation"))
        
        visualize_component(**last_args, show=True)
        w_save.disabled = False

w_match.observe(render_selected_node)

# Search logic: Loads files and finds all matches
def on_search(b):
    w_save.disabled = True
    w_match.disabled = True
    w_match.options = []
    
    with out_plot:
        clear_output(wait=True)
        if not w_file.value or not w_graph.value: return print("Select TSV and Graph.")
        
        print("Scanning files...")
        df = pd.read_csv(RESULTS_DIR / w_file.value, sep='\t')
        state['node_probs'] = {(r['file_name'], r['roi_id']): r['node_prob'] for _, r in df.iterrows()}
        state['graph'] = torch.load(GRAPHS_DIR / w_graph.value, map_location='cpu')
        state['G'] = create_nx_graph(state['graph'], ionization_mode=IONIZATION_MODE)
        
        mz_feat, rt_feat_sec = state['graph']["node_features"][:, 0], state['graph']["node_features"][:, 1]
        matches = torch.where((torch.abs(mz_feat - w_mz.value) <= (w_mz.value * w_ppm.value / 1e6)) & 
                              (torch.abs(rt_feat_sec - (w_rt.value * 60.0)) <= 5.0))[0]
        
        if not len(matches): return print("❌ No matches found. Try widening your tolerances.")
        
        # Build dropdown options
        options = []
        for m in matches:
            idx = m.item()
            m_mz = mz_feat[idx].item()
            m_rt = rt_feat_sec[idx].item() / 60.0
            options.append((f"Node {idx} (m/z: {m_mz:.4f}, RT: {m_rt:.2f}m)", idx))
        
        print(f"🎯 Found {len(options)} match(es).")
        
        w_match.options = options
        w_match.disabled = False

def on_save(b):
    with out_plot:
        base = w_graph.value.replace("graph_", "").replace(".pt", "")
        node_id = w_match.value
        path = VIS_DIR / f"{base}_{w_title.value}_{w_mz.value}_{w_rt.value}_node{node_id}.html"
        visualize_component(**last_args, save_html=str(path), show=False)
        print(f"💾 Saved to: {path}")

w_btn.on_click(on_search)
w_save.on_click(on_save)

ui = widgets.VBox([
    widgets.HBox([w_file, w_graph]), 
    widgets.HBox([w_mz, w_rt, w_ppm]), 
    widgets.HBox([w_btn, w_match]),
    widgets.HBox([w_title, w_save])
])
ui.add_class('search-card')
display(ui, out_plot)

Output()